# Differentially Private Random Forest - Gallstone Dataset\n\nTrains a DP Random Forest model (via IBM diffprivlib) on the Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
from diffprivlib.models import RandomForestClassifier as DPRandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
bounds = data['bounds']


## 3. Load Baseline Results


In [3]:
try:
    baseline_df = pd.read_csv('output/baseline_results.csv')
    baseline_accuracy = baseline_df['accuracy'].values[0]
    baseline_f1 = baseline_df['f1_score'].values[0]
except FileNotFoundError:
    baseline_accuracy = None
    baseline_f1 = None


## 4. Sweep Epsilon


In [4]:
epsilon_values = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 4.0, 6.0, 8.0, 10.0]
N_RUNS = 30

results = {
    'epsilon': [], 'accuracy_mean': [], 'accuracy_std': [],
    'f1_score_mean': [], 'f1_score_std': [],
    'precision_mean': [], 'precision_std': [],
    'recall_mean': [], 'recall_std': [],
}

all_results = {
    'model_type': 'Differentially Private Random Forest',
    'dataset': 'Gallstone',
    'n_runs_per_epsilon': N_RUNS,
    'epsilon_results': {}
}

print(f"Training DP RF with {N_RUNS} runs per epsilon...")
print('=' * 80)

for eps in epsilon_values:
    run_accs, run_f1s, run_precs, run_recs, run_cms = [], [], [], [], []

    extra = ExtraMetrics()
    for run in range(N_RUNS):
        seed = run * 10 + 42
        dp_rf = DPRandomForestClassifier(
            n_estimators=20, epsilon=eps, random_state=seed,
            n_jobs=-1, max_depth=4, bounds=bounds
        )
        _t0 = time.perf_counter()
        dp_rf.fit(X_train, y_train)
        train_time = time.perf_counter() - _t0
        y_pred = dp_rf.predict(X_test)
        
        run_accs.append(accuracy_score(y_test, y_pred))
        extra.add(y_test, y_pred, model=dp_rf, X=X_test, train_time=train_time)
        run_f1s.append(f1_score(y_test, y_pred, zero_division=0))
        run_precs.append(precision_score(y_test, y_pred, zero_division=0))
        run_recs.append(recall_score(y_test, y_pred, zero_division=0))
        run_cms.append(confusion_matrix(y_test, y_pred).tolist())
        if run == 0:
            import sys
            import os
            sys.path.append(os.path.abspath(os.path.join('..', '..')))
            from model_exporter import export_model
            export_model(dp_rf, f'output/model/dp_rf_model_eps_{eps}.pkl')

    acc_mean, acc_std = np.mean(run_accs), np.std(run_accs)
    f1_mean, f1_std = np.mean(run_f1s), np.std(run_f1s)
    prec_mean, prec_std = np.mean(run_precs), np.std(run_precs)
    rec_mean, rec_std = np.mean(run_recs), np.std(run_recs)

    results['epsilon'].append(eps)
    # Merge in the added metrics; keys the notebook already tracks are skipped.
    for _key, _value in extra.summary().items():
        results.setdefault(_key, []).append(_value)
    results['accuracy_mean'].append(acc_mean)
    results['accuracy_std'].append(acc_std)
    results['f1_score_mean'].append(f1_mean)
    results['f1_score_std'].append(f1_std)
    results['precision_mean'].append(prec_mean)
    results['precision_std'].append(prec_std)
    results['recall_mean'].append(rec_mean)
    results['recall_std'].append(rec_std)

    all_results['epsilon_results'][str(eps)] = {
        'epsilon': float(eps), 'n_runs': N_RUNS,
        'accuracy_mean': acc_mean, 'accuracy_std': acc_std,
        'f1_mean': f1_mean, 'f1_std': f1_std,
        'precision_mean': prec_mean, 'precision_std': prec_std,
        'recall_mean': rec_mean, 'recall_std': rec_std,
        'per_run_accuracies': run_accs, 'per_run_f1s': run_f1s,
        'per_run_confusion_matrices': run_cms
    }
    all_results['epsilon_results'][str(eps)].update(extra.summary())
    all_results['epsilon_results'][str(eps)].update(extra.per_run())

    print(f"eps={eps:5.2f} | Acc: {acc_mean:.4f}+/-{acc_std:.4f} | F1: {f1_mean:.4f}+/-{f1_std:.4f}")

print('=' * 80)


Training DP RF with 30 runs per epsilon...


Model successfully exported to output/model/dp_rf_model_eps_0.1.pkl


eps= 0.10 | Acc: 0.5208+/-0.0597 | F1: 0.4333+/-0.2071
Model successfully exported to output/model/dp_rf_model_eps_0.2.pkl


eps= 0.20 | Acc: 0.5292+/-0.0499 | F1: 0.4380+/-0.1932
Model successfully exported to output/model/dp_rf_model_eps_0.4.pkl


eps= 0.40 | Acc: 0.5396+/-0.0636 | F1: 0.3891+/-0.2439
Model successfully exported to output/model/dp_rf_model_eps_0.6.pkl


...

eps= 2.00 | Acc: 0.5510+/-0.0624 | F1: 0.4030+/-0.2068
Model successfully exported to output/model/dp_rf_model_eps_4.0.pkl


eps= 4.00 | Acc: 0.5490+/-0.0548 | F1: 0.3766+/-0.1927
Model successfully exported to output/model/dp_rf_model_eps_6.0.pkl


eps= 6.00 | Acc: 0.5500+/-0.0563 | F1: 0.3756+/-0.1968
Model successfully exported to output/model/dp_rf_model_eps_8.0.pkl


eps= 8.00 | Acc: 0.5490+/-0.0560 | F1: 0.3738+/-0.1980
Model successfully exported to output/model/dp_rf_model_eps_10.0.pkl


eps=10.00 | Acc: 0.5490+/-0.0560 | F1: 0.3738+/-0.1980


## 5. Save Results


In [5]:
os.makedirs('output', exist_ok=True)
results_df = pd.DataFrame(results)
if baseline_accuracy is not None:
    results_df['accuracy_loss_mean'] = 1 - results_df['accuracy_mean'] / baseline_accuracy
    results_df['accuracy_loss_pct'] = results_df['accuracy_loss_mean'] * 100

results_df.to_csv('output/dp_results.csv', index=False)
with open('output/dp_rf_report.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print("Results saved.")


Results saved.
